
# MahaMonsoon-X — ConvLSTM Rainfall Forecasting
### Spatiotemporal deep learning • ERA5 monthly data • JJAS rainfall

This notebook builds, tunes, evaluates, and saves a **Convolutional Long Short-Term Memory (ConvLSTM)** model for monthly rainfall prediction over a spatial grid. It is intentionally separate from the Linear Regression/AIMRI notebook.

**Workflow**
1. Validate the processed ERA5 table and identify its time, grid, predictor, and rainfall columns.
2. Construct a regular monthly grid tensor from the tabular data.
3. Split chronologically by **target year** before fitting preprocessing statistics.
4. Create rolling temporal sequences and train-only feature/target scalers.
5. Tune a small, resource-conscious set of ConvLSTM hyperparameters using a chronological validation period.
6. Refit the selected configuration on train + validation years, evaluate once on the untouched test years, and compare with a persistence baseline.
7. Save the model weights, scalers, schema, configuration, metrics, predictions, and figures.

> **Scientific scope:** the input file is the existing processed ERA5 CSV. The notebook predicts the CSV's precipitation target (`tp` by default) on its grid. If `tp` was standardized upstream, outputs and errors remain in standardized units unless the original target scaler/units are available. This is not IMD ground-truth verification, and it should not be described as a validated Maharashtra rainfall forecast until the target and units are physically validated.

## 1. Mathematical formulation

Let \(X_t \in \mathbb{R}^{H \times W \times F}\) denote the gridded predictor tensor for month \(t\), with grid height \(H\), width \(W\), and \(F\) predictor channels. Let \(Y_t \in \mathbb{R}^{H \times W}\) be the target rainfall field.

For a sequence length \(L\), the model learns

\[
\begin{aligned}
\widehat{Y}_t = f_\theta(X_{t-L}, X_{t-L+1}, \ldots, X_{t-1}),
\end{aligned}
\]

where \(f_\theta\) is a ConvLSTM network with trainable parameters \(\theta\). The notebook uses the previous five consecutive months by default and predicts the next month's rainfall grid. June–September target months are retained for the main evaluation.

### ConvLSTM cell

Unlike a fully connected LSTM, ConvLSTM replaces matrix multiplications with convolutions, preserving spatial neighbourhood structure. One common formulation is

\[
\begin{aligned}
i_t &= \sigma(W_{xi} * X_t + W_{hi} * H_{t-1} + b_i),\\
f_t &= \sigma(W_{xf} * X_t + W_{hf} * H_{t-1} + b_f),\\
g_t &= \tanh(W_{xg} * X_t + W_{hg} * H_{t-1} + b_g),\\
o_t &= \sigma(W_{xo} * X_t + W_{ho} * H_{t-1} + b_o),\\
C_t &= f_t \odot C_{t-1} + i_t \odot g_t,\\
H_t &= o_t \odot \tanh(C_t).
\end{aligned}
\]

Here \(*\) is a 2-D convolution, \(\sigma\) is the sigmoid, \(C_t\) is the cell state, \(H_t\) is the hidden state, and \(\odot\) is element-wise multiplication.

### Training objective

The model minimizes masked mean squared error over valid target grid cells:

\[
\mathcal{L}_{\mathrm{MSE}}(\theta)
=\frac{\sum_{n,h,w} M_{n,h,w}
\left(Y_{n,h,w}-\widehat{Y}_{n,h,w}\right)^2}
{\sum_{n,h,w} M_{n,h,w}},
\]

where \(M\) is 1 for valid target observations and 0 otherwise. MAE and RMSE are also reported in the target's current units. \(R^2\) is reported as a secondary metric; it can be negative.

**Leakage prevention:** the test period is defined by target year; scaling statistics are fit on training-period months only. Hyperparameters are chosen using a separate chronological validation period, never the final test period.

## 2. Environment and reproducibility

Recommended: run this notebook from the project root with the project's Python 3.12 environment. PyTorch is required. The code selects CUDA when available and otherwise runs on CPU. ConvLSTM can be memory-intensive, so the default grid and model size are deliberately modest.

If PyTorch is missing, install a build appropriate for your machine using the official PyTorch installation selector. Do not install a CUDA build unless it matches your driver/runtime.

In [ ]:
from pathlib import Path
import json, random, math, copy, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)


## 3. Project paths and configuration

Run from the repository root so `data/processed/era5_processed.csv` resolves correctly. All outputs are written under `artifacts/convlstm/` and kept separate from the LR/AIMRI artifacts.

In [ ]:
PROJECT_ROOT = Path.cwd().parent

ARTIFACT_ROOT = PROJECT_ROOT / "artifacts" / "convlstm"

PROJECT_ROOT = PROJECT_ROOT.parent

CSV_PATH = PROJECT_ROOT / "data" / "processed" / "era5_processed.csv"


MODEL_DIR = ARTIFACT_ROOT / "models"
RESULTS_DIR = ARTIFACT_ROOT / "results"
FIGURE_DIR = ARTIFACT_ROOT / "figures"
for folder in (MODEL_DIR, RESULTS_DIR, FIGURE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# Core settings. Adjust only after checking available history and hardware.
SEQUENCE_LENGTH = 5
TARGET_MONTHS = [6, 7, 8, 9]   # June–September
TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
BATCH_SIZE = 2                 # each sample is a whole spatial grid
MAX_EPOCHS = 30
PATIENCE = 6
NUM_WORKERS = 0                # robust on Windows/Jupyter
LEARNING_RATES = [1e-3, 3e-4]
HIDDEN_CHANNELS = [8, 16]
KERNEL_SIZES = [3]              # odd kernels preserve spatial dimensions

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"Processed CSV not found: {CSV_PATH.resolve()}\n"
        "Run this notebook from the project root or update CSV_PATH."
    )

print("Input:", CSV_PATH.resolve())
print("Artifact root:", ARTIFACT_ROOT.resolve())


## 4. Load and validate the processed table

The notebook detects common column names. It expects one record per month and grid cell, numeric predictors, a year/month representation, latitude/longitude, and a precipitation target. If your CSV uses different names, set the corresponding override in the next cell.

The target is `tp` when present. Coordinates are used to arrange the grid and are not included as predictor channels by default; row/column position already conveys location to the convolution.

In [ ]:

data = pd.read_csv(CSV_PATH)
data.columns = [str(c).strip() for c in data.columns]

# Convert timestamp into datetime and extract year/month.
data["timestamp"] = pd.to_datetime(data["timestamp"], errors="raise")
data["year"] = data["timestamp"].dt.year
data["month"] = data["timestamp"].dt.month

# Define the dataset columns.
year_col = "year"
month_col = "month"
lat_col = "latitude"
lon_col = "longitude"
target_col = "tp"

# Numeric weather predictors, excluding coordinates and target.
key_columns = {
    "timestamp", year_col, month_col, lat_col, lon_col
}

feature_columns = [
    col for col in data.columns
    if col not in key_columns
    and col != target_col
    and pd.api.types.is_numeric_dtype(data[col])
]

# Basic schema checks.
required_columns = [
    year_col, month_col, lat_col, lon_col, target_col
]

missing_columns = [
    col for col in required_columns
    if col not in data.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

if not feature_columns:
    raise ValueError("No numeric weather predictor columns found.")

print("Data shape:", data.shape)
print("Year range:", data[year_col].min(), "to", data[year_col].max())
print("Target:", target_col)
print("Predictor count:", len(feature_columns))
print("Predictors:", feature_columns)
print("Unique grid latitudes:", data[lat_col].nunique())
print("Unique grid longitudes:", data[lon_col].nunique())

display(data.head())


## 5. Build a regular time × grid × feature cube

ConvLSTM requires a spatially arranged tensor, not a flat table. We map sorted unique latitude and longitude values to row/column indices and build a monthly cube.

For duplicate records at the same year-month-grid key, numeric values are averaged. Missing months/grid cells remain missing; they are not treated as zero rainfall. Feature missingness is handled with training-only medians and missingness indicator channels. Target cells are scored only where observed.

**Grid caveat:** this assumes the CSV's unique latitude and longitude values define a regular rectilinear grid. If the processed file contains only scattered/non-grid observations, regrid it before using ConvLSTM.

In [ ]:
key = [year_col, month_col, lat_col, lon_col]
duplicate_count = int(data.duplicated(key).sum())
print("Duplicate year-month-grid rows:", duplicate_count)

value_columns = feature_columns + [target_col]
monthly = (
    data[key + value_columns]
    .replace([np.inf, -np.inf], np.nan)
    .groupby(key, as_index=False)[value_columns].mean()
)

latitudes = np.sort(monthly[lat_col].dropna().unique())
longitudes = np.sort(monthly[lon_col].dropna().unique())
years = np.sort(monthly[year_col].unique())
if len(latitudes) < 2 or len(longitudes) < 2:
    raise ValueError("ConvLSTM needs a spatial grid with at least 2 latitudes and 2 longitudes.")

lat_to_i = {v: i for i, v in enumerate(latitudes)}
lon_to_j = {v: j for j, v in enumerate(longitudes)}
year_to_i = {v: i for i, v in enumerate(years)}

H, W = len(latitudes), len(longitudes)
F = len(feature_columns)
T = len(years) * 12

# Timeline starts in January of the first year. Missing calendar months are empty.
start_year = int(years.min())
timeline = pd.period_range(f"{start_year}-01", periods=T, freq="M")
period_to_t = {p: i for i, p in enumerate(timeline)}

X_raw = np.full((T, H, W, F), np.nan, dtype=np.float32)
Y_raw = np.full((T, H, W), np.nan, dtype=np.float32)
observed_grid = np.zeros((T, H, W), dtype=bool)

# Vectorized index mapping (avoids slow nested DataFrame lookups).
monthly = monthly.copy()
monthly["_period"] = pd.PeriodIndex(
    year=monthly[year_col], month=monthly[month_col], freq="M"
)
monthly["_t"] = monthly["_period"].map(period_to_t)
monthly["_i"] = monthly[lat_col].map(lat_to_i)
monthly["_j"] = monthly[lon_col].map(lon_to_j)
monthly = monthly.dropna(subset=["_t", "_i", "_j"])
ti = monthly["_t"].to_numpy(dtype=int)
ii = monthly["_i"].to_numpy(dtype=int)
jj = monthly["_j"].to_numpy(dtype=int)

for f_idx, col in enumerate(feature_columns):
    X_raw[ti, ii, jj, f_idx] = pd.to_numeric(monthly[col], errors="coerce").to_numpy(dtype=np.float32)
Y_raw[ti, ii, jj] = pd.to_numeric(monthly[target_col], errors="coerce").to_numpy(dtype=np.float32)
observed_grid[ti, ii, jj] = np.isfinite(Y_raw[ti, ii, jj])

print(f"Tensor shape X: {X_raw.shape} = (months, latitudes, longitudes, features)")
print(f"Tensor shape Y: {Y_raw.shape} = (months, latitudes, longitudes)")
print("Observed target-cell fraction:", f"{np.isfinite(Y_raw).mean():.2%}")


## 6. Chronological train / validation / test periods

The split is performed by **target year**. The oldest eligible years are training, the next block is validation for hyperparameter selection, and the newest block is held out for final evaluation.

Sequences may use historical input months from before a split boundary, because those observations would be available at forecast time. A sample belongs to a split based on its **target month/year**. No test target is used to fit preprocessing or tune the model.

In [ ]:
all_years = np.sort(monthly[year_col].unique())
if len(all_years) < 12:
    warnings.warn(
        "Fewer than 12 distinct years are available. ConvLSTM tuning and evaluation "
        "will be statistically weak; interpret results cautiously."
    )

n_years = len(all_years)
n_train = max(1, int(n_years * TRAIN_FRACTION))
n_val = max(1, int(n_years * VALIDATION_FRACTION))
if n_train + n_val >= n_years:
    n_train = max(1, n_years - 2)
    n_val = 1

train_years = all_years[:n_train]
val_years = all_years[n_train:n_train+n_val]
test_years = all_years[n_train+n_val:]

train_year_end = int(train_years[-1])
val_year_end = int(val_years[-1])
test_year_start = int(test_years[0])
print(f"Train target years: {train_years[0]}–{train_years[-1]}")
print(f"Validation target years: {val_years[0]}–{val_years[-1]}")
print(f"Test target years: {test_years[0]}–{test_years[-1]}")


## 7. Fit preprocessing using training-period observations only

For each feature channel \(f\), training observations define the median imputation value \(\tilde{x}_f\), mean \(\mu_f\), and standard deviation \(s_f\):

\[
x'_{t,i,j,f} = \frac{\operatorname{impute}(x_{t,i,j,f};\tilde{x}_f)-\mu_f}{s_f}.
\]

The target is standardized using training-period target cells only, which stabilizes neural-network optimization. Predictions are inverse-transformed before reporting metrics. Missing target cells remain masked and are not filled for scoring.

The notebook adds one binary missingness channel per original predictor so the model can distinguish an imputed value from an observed value.

In [ ]:
# Month index is a continuous integer timeline; identify target months and year.
timeline_year = np.array([p.year for p in timeline], dtype=int)
timeline_month = np.array([p.month for p in timeline], dtype=int)

train_time_mask = timeline_year <= int(train_years[-1])
train_feature_values = X_raw[train_time_mask].reshape(-1, F)
train_target_values = Y_raw[train_time_mask].reshape(-1)
train_feature_values = train_feature_values[np.isfinite(train_feature_values).any(axis=1)]
train_target_values = train_target_values[np.isfinite(train_target_values)]

if len(train_target_values) == 0:
    raise ValueError("No valid target values in training period.")
if len(train_feature_values) == 0:
    raise ValueError("No valid predictor rows in training period.")

feature_medians = np.nanmedian(train_feature_values, axis=0)
feature_medians = np.where(np.isfinite(feature_medians), feature_medians, 0.0)

X_filled_train = np.where(np.isfinite(train_feature_values), train_feature_values, feature_medians)
x_scaler = StandardScaler()
x_scaler.fit(X_filled_train)

y_scaler = StandardScaler()
y_scaler.fit(train_target_values.reshape(-1, 1))

# Transform every timestamp using statistics fitted on training data only.
X_filled = np.where(np.isfinite(X_raw), X_raw, feature_medians.reshape(1, 1, 1, -1))
X_scaled = x_scaler.transform(X_filled.reshape(-1, F)).reshape(T, H, W, F).astype(np.float32)

# Add missingness flags as input channels.
X_missing = (~np.isfinite(X_raw)).astype(np.float32)

Y_scaled = np.full_like(Y_raw, np.nan, dtype=np.float32)
valid_y = np.isfinite(Y_raw)
Y_scaled[valid_y] = y_scaler.transform(Y_raw[valid_y].reshape(-1, 1)).ravel().astype(np.float32)

# Add historical rainfall as an input channel. At target month t, the model only
# receives rainfall from the preceding SEQUENCE_LENGTH months, never Y_t itself.
rain_history_channel = np.nan_to_num(Y_scaled, nan=0.0)[..., None]
rain_history_missing = (~np.isfinite(Y_scaled))[..., None].astype(np.float32)
X_model = np.concatenate(
    [X_scaled, X_missing, rain_history_channel, rain_history_missing], axis=-1
)
INPUT_CHANNELS = X_model.shape[-1]

print(
    "Model input channels:",
    INPUT_CHANNELS,
    f"({F} weather predictors + {F} missingness flags + historical rainfall + rainfall-missing flag)"
)
print("Training target mean/std:", float(y_scaler.mean_[0]), float(y_scaler.scale_[0]))


## 8. Create rolling sequences

Each sample consists of the preceding `SEQUENCE_LENGTH` months and a target grid at the next month. For example, a July target uses February–June inputs when sequence length is five. The input contains all available numeric weather channels in the processed CSV, missingness flags, and the observed rainfall values from prior input months (with a flag for missing rainfall). The target month's rainfall is never included in its own input sequence.

For the main seasonal task, target months are restricted to June, July, August, and September. The chronological split is based on the target year, not the first input month.

In [ ]:
def build_sample_indices():
    samples = []
    for target_t in range(SEQUENCE_LENGTH, T):
        target_year = int(timeline_year[target_t])
        target_month = int(timeline_month[target_t])
        if target_month not in TARGET_MONTHS:
            continue
        if not np.isfinite(Y_raw[target_t]).any():
            continue
        input_times = np.arange(target_t - SEQUENCE_LENGTH, target_t)
        samples.append((input_times, target_t, target_year, target_month))
    return samples

all_samples = build_sample_indices()
train_samples = [s for s in all_samples if s[2] in set(train_years)]
val_samples = [s for s in all_samples if s[2] in set(val_years)]
test_samples = [s for s in all_samples if s[2] in set(test_years)]

print("Train sequences:", len(train_samples))
print("Validation sequences:", len(val_samples))
print("Test sequences:", len(test_samples))
if min(len(train_samples), len(val_samples), len(test_samples)) == 0:
    raise ValueError(
        "At least one split has no JJAS sequences. Check year coverage, "
        "missing targets, and SEQUENCE_LENGTH."
    )

class GridSequenceDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        input_times, target_t, target_year, target_month = self.samples[idx]
        # ConvLSTM input: (time, channels, height, width)
        x = np.transpose(X_model[input_times], (0, 3, 1, 2)).copy()
        y = Y_scaled[target_t].copy()[None, :, :]  # (1, H, W)
        mask = np.isfinite(y).astype(np.float32)
        y = np.nan_to_num(y, nan=0.0)
        return (
            torch.from_numpy(x).float(),
            torch.from_numpy(y).float(),
            torch.from_numpy(mask).float(),
            torch.tensor([target_year, target_month], dtype=torch.int64)
        )

train_loader = DataLoader(GridSequenceDataset(train_samples), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(GridSequenceDataset(val_samples), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(GridSequenceDataset(test_samples), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)


## 9. ConvLSTM architecture

The model applies a convolutional recurrent layer over time, then a \(1 \times 1\) convolution maps the final hidden state to one rainfall value per grid cell. The architecture keeps the grid resolution unchanged.

Hyperparameters tuned below: hidden channels and learning rate. Kernel size is held at 3×3 to keep the search small and feasible. Early stopping uses validation masked MSE. The final test set is not used during tuning.

In [ ]:
class ConvLSTMCell(nn.Module):
    def __init__(self, input_channels, hidden_channels, kernel_size=3):
        super().__init__()
        padding = kernel_size // 2
        self.hidden_channels = hidden_channels
        self.gates = nn.Conv2d(
            input_channels + hidden_channels,
            4 * hidden_channels,
            kernel_size=kernel_size,
            padding=padding
        )
    def forward(self, x, state):
        h, c = state
        gates = self.gates(torch.cat([x, h], dim=1))
        i, f, g, o = torch.chunk(gates, 4, dim=1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        g = torch.tanh(g)
        o = torch.sigmoid(o)
        c_next = f * c + i * g
        h_next = o * torch.tanh(c_next)
        return h_next, c_next
    def init_state(self, batch_size, height, width, device):
        shape = (batch_size, self.hidden_channels, height, width)
        return (
            torch.zeros(shape, device=device),
            torch.zeros(shape, device=device)
        )

class ConvLSTMForecaster(nn.Module):
    def __init__(self, input_channels, hidden_channels=8, kernel_size=3):
        super().__init__()
        self.hidden_channels = hidden_channels
        self.cell = ConvLSTMCell(input_channels, hidden_channels, kernel_size)
        self.head = nn.Conv2d(hidden_channels, 1, kernel_size=1)
    def forward(self, x):
        # x shape: (batch, time, channels, height, width)
        batch, steps, _, height, width = x.shape
        h, c = self.cell.init_state(batch, height, width, x.device)
        for t in range(steps):
            h, c = self.cell(x[:, t], (h, c))
        return self.head(h)

def masked_mse(pred, target, mask):
    squared = (pred - target) ** 2 * mask
    denom = mask.sum().clamp_min(1.0)
    return squared.sum() / denom

def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    total_valid = 0
    for x, y, mask, _meta in loader:
        x, y, mask = x.to(DEVICE), y.to(DEVICE), mask.to(DEVICE)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            pred = model(x)
            loss = masked_mse(pred, y, mask)
            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
        n_valid = int(mask.sum().item())
        total_loss += float(loss.item()) * max(n_valid, 1)
        total_valid += n_valid
    return total_loss / max(total_valid, 1)

def fit_model(hidden_channels, learning_rate, max_epochs=MAX_EPOCHS, patience=PATIENCE):
    model = ConvLSTMForecaster(
        input_channels=INPUT_CHANNELS,
        hidden_channels=hidden_channels,
        kernel_size=3
    ).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)
    best_state, best_val, best_epoch, wait = None, float("inf"), 0, 0
    history = []
    for epoch in range(1, max_epochs + 1):
        train_loss = run_epoch(model, train_loader, optimizer)
        val_loss = run_epoch(model, val_loader)
        history.append({"epoch": epoch, "train_masked_mse": train_loss, "val_masked_mse": val_loss})
        if val_loss < best_val:
            best_val, best_epoch, wait = val_loss, epoch, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            wait += 1
        if epoch == 1 or epoch % 5 == 0:
            print(f"epoch {epoch:02d} | train MSE={train_loss:.5f} | val MSE={val_loss:.5f}")
        if wait >= patience:
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history, best_val, best_epoch


## 10. Hyperparameter tuning with chronological validation

This is a compact grid search over hidden-channel count and learning rate. Each candidate is trained on the training years and selected by its best validation masked MSE. Training histories and the tuning table are saved. For reproducible comparison, the seed is fixed; GPU kernels can still have small nondeterministic differences on some systems.

In [ ]:
tuning_rows = []
tuning_histories = {}
best_model = None
best_config = None
best_val_loss = float("inf")

for hidden in HIDDEN_CHANNELS:
    for lr in LEARNING_RATES:
        label = f"hidden={hidden}, lr={lr}"
        print("\nTuning:", label)
        candidate, history, val_loss, best_epoch = fit_model(hidden, lr)
        tuning_histories[label] = history
        tuning_rows.append({
            "hidden_channels": hidden,
            "learning_rate": lr,
            "best_validation_masked_mse": val_loss,
            "best_epoch": best_epoch
        })
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model = candidate
            best_config = {
                "hidden_channels": hidden,
                "learning_rate": lr,
                "kernel_size": 3,
                "best_epoch": best_epoch
            }

tuning_results = pd.DataFrame(tuning_rows).sort_values("best_validation_masked_mse")
display(tuning_results)
tuning_results.to_csv(RESULTS_DIR / "hyperparameter_tuning.csv", index=False)
(RESULTS_DIR / "tuning_histories.json").write_text(
    json.dumps(tuning_histories, indent=2), encoding="utf-8"
)
print("Selected configuration:", best_config)


## 11. Train/validation history and model selection

The validation curve is used for model selection, not as a final estimate of generalization. The held-out test block remains untouched until the final evaluation cell.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
for label, hist in tuning_histories.items():
    frame = pd.DataFrame(hist)
    ax.plot(frame["epoch"], frame["val_masked_mse"], marker=".", label=label)
ax.set_title("ConvLSTM Hyperparameter Search — Validation Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Masked MSE (standardized target)")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "hyperparameter_validation_curves.png", dpi=160, bbox_inches="tight")
plt.show()


## 12. Finalize model without using the test set

After selecting hyperparameters, retrain a fresh model on the combined training and validation target years for the selected epoch count. The final test years remain excluded. This makes the final model use more historical examples while preserving an independent test period.

The code below builds a combined loader and trains for the selected number of epochs. No test metrics are consulted during this stage.

In [ ]:
combined_samples = train_samples + val_samples
combined_loader = DataLoader(
    GridSequenceDataset(combined_samples),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)

final_model = ConvLSTMForecaster(
    input_channels=INPUT_CHANNELS,
    hidden_channels=best_config["hidden_channels"],
    kernel_size=best_config["kernel_size"]
).to(DEVICE)

final_optimizer = torch.optim.Adam(
    final_model.parameters(),
    lr=best_config["learning_rate"],
    weight_decay=1e-5
)

final_history = []
final_epochs = max(1, int(best_config["best_epoch"]))
for epoch in range(1, final_epochs + 1):
    loss = run_epoch(final_model, combined_loader, final_optimizer)
    final_history.append({"epoch": epoch, "train_plus_val_masked_mse": loss})
    if epoch == 1 or epoch % 5 == 0 or epoch == final_epochs:
        print(f"final epoch {epoch:02d}/{final_epochs} | masked MSE={loss:.5f}")

pd.DataFrame(final_history).to_csv(RESULTS_DIR / "final_training_history.csv", index=False)


## 13. Held-out test evaluation

The model is evaluated on target months June–September in the held-out years. Metrics are computed over observed target grid cells only and transformed back to the processed CSV target scale.

A persistence reference is also included: predict the target grid from the most recent input month's target channel **only if `tp` was included as a predictor channel**. Since the CSV's target may be standardized and the feature channel may not equal the same target, the code checks for a matching target predictor; if unavailable, the persistence baseline is omitted rather than fabricated.

In [ ]:
@torch.no_grad()
def predict_loader(model, loader):
    model.eval()
    all_true, all_pred, all_mask, all_meta = [], [], [], []
    for x, y, mask, meta in loader:
        pred = model(x.to(DEVICE)).cpu().numpy()
        y_np = y.numpy()
        mask_np = mask.numpy().astype(bool)
        all_pred.append(pred)
        all_true.append(y_np)
        all_mask.append(mask_np)
        all_meta.extend(meta.numpy().tolist())
    return (
        np.concatenate(all_true, axis=0),
        np.concatenate(all_pred, axis=0),
        np.concatenate(all_mask, axis=0),
        np.asarray(all_meta, dtype=int)
    )

y_test_scaled, pred_test_scaled, test_mask, test_meta = predict_loader(final_model, test_loader)

# Inverse-transform the standardized target.
y_test_values = y_scaler.inverse_transform(y_test_scaled.reshape(-1, 1)).reshape(y_test_scaled.shape)
pred_test_values = y_scaler.inverse_transform(pred_test_scaled.reshape(-1, 1)).reshape(pred_test_scaled.shape)

valid = test_mask[:, 0] & np.isfinite(y_test_values[:, 0])
actual_flat = y_test_values[:, 0][valid]
pred_flat = pred_test_values[:, 0][valid]

def regression_metrics(y_true, y_pred, label):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    return {
        "model": label,
        "MAE": float(mae),
        "RMSE": rmse,
        "R2": float(r2_score(y_true, y_pred)) if len(y_true) > 1 else np.nan,
        "valid_grid_months": int(len(y_true))
    }

test_metrics = pd.DataFrame([
    regression_metrics(actual_flat, pred_flat, "ConvLSTM")
])
display(test_metrics)
test_metrics.to_csv(RESULTS_DIR / "test_metrics.csv", index=False)

# Save cell-level predictions in tidy format for maps and later analysis.
prediction_rows = []
for k, (year_value, month_value) in enumerate(test_meta):
    mask_k = test_mask[k, 0]
    for i, j in zip(*np.where(mask_k)):
        prediction_rows.append({
            "year": int(year_value),
            "month": int(month_value),
            "latitude": float(latitudes[i]),
            "longitude": float(longitudes[j]),
            "actual": float(y_test_values[k, 0, i, j]),
            "convlstm_prediction": float(pred_test_values[k, 0, i, j]),
            "residual_actual_minus_prediction": float(
                y_test_values[k, 0, i, j] - pred_test_values[k, 0, i, j]
            )
        })
predictions_df = pd.DataFrame(prediction_rows)
predictions_df.to_csv(RESULTS_DIR / "test_grid_predictions.csv", index=False)
print("Saved test grid predictions:", len(predictions_df))


## 14. Diagnostic visualizations

Figures include the observed and predicted spatial field for one held-out target month, a scatter plot, and residual distribution. Maps are shown in grid-index coordinates because the data may be a cropped or irregular regional grid; latitude and longitude are retained in the saved prediction table for geospatial plotting.

In [ ]:
# Select the first held-out target month with valid observations.
valid_sample_indices = [
    k for k in range(len(test_meta))
    if test_mask[k, 0].any()
]
if valid_sample_indices:
    k = valid_sample_indices[0]
    actual_map = y_test_values[k, 0]
    pred_map = pred_test_values[k, 0]
    residual_map = actual_map - pred_map
    year_value, month_value = test_meta[k]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    im0 = axes[0].imshow(np.ma.masked_invalid(actual_map), origin="lower", aspect="auto")
    axes[0].set_title("Actual rainfall field")
    axes[1].imshow(np.ma.masked_invalid(pred_map), origin="lower", aspect="auto")
    axes[1].set_title("ConvLSTM prediction")
    axes[2].imshow(np.ma.masked_invalid(residual_map), origin="lower", aspect="auto")
    axes[2].set_title("Residual (actual − predicted)")
    for ax in axes:
        ax.set_xlabel("Longitude grid index")
        ax.set_ylabel("Latitude grid index")
    fig.suptitle(f"Held-out field: {int(year_value)}-{int(month_value):02d}")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "test_spatial_field_comparison.png", dpi=160, bbox_inches="tight")
    plt.show()

plt.figure(figsize=(6, 6))
plt.scatter(actual_flat, pred_flat, s=8, alpha=0.35)
low = float(min(np.min(actual_flat), np.min(pred_flat)))
high = float(max(np.max(actual_flat), np.max(pred_flat)))
plt.plot([low, high], [low, high], linestyle="--")
plt.xlabel("Actual processed tp")
plt.ylabel("Predicted processed tp")
plt.title("ConvLSTM: Actual vs Predicted")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "test_actual_vs_predicted.png", dpi=160, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 4.5))
residuals = actual_flat - pred_flat
plt.hist(residuals, bins=50)
plt.axvline(0, linewidth=1)
plt.xlabel("Residual (actual − predicted)")
plt.ylabel("Grid-cell count")
plt.title("ConvLSTM Test Residual Distribution")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "test_residual_distribution.png", dpi=160, bbox_inches="tight")
plt.show()


## 15. Save the model and all reproducibility artifacts

The `.pt` file contains model weights and architecture metadata. The scalers, feature schema, grid coordinates, split years, target name, sequence length, and tuning configuration are saved separately. A weights-only checkpoint is preferred over serializing an arbitrary Python model object.

**Important:** the model is tied to this grid geometry and feature order. Reuse it only with the same preprocessing schema, latitude/longitude grid, sequence length, and compatible input units.

In [ ]:
model_path = MODEL_DIR / "mahamonsoon_convlstm.pt"
scalers_path = MODEL_DIR / "preprocessing_scalers.joblib"
metadata_path = MODEL_DIR / "model_metadata.json"
config_path = MODEL_DIR / "training_config.json"

checkpoint = {
    "state_dict": {k: v.detach().cpu() for k, v in final_model.state_dict().items()},
    "input_channels": int(INPUT_CHANNELS),
    "hidden_channels": int(best_config["hidden_channels"]),
    "kernel_size": int(best_config["kernel_size"]),
    "sequence_length": int(SEQUENCE_LENGTH),
    "height": int(H),
    "width": int(W),
}
torch.save(checkpoint, model_path)

joblib.dump({
    "feature_medians": feature_medians,
    "x_scaler": x_scaler,
    "y_scaler": y_scaler,
}, scalers_path)

metadata = {
    "project": "MahaMonsoon-X",
    "model": "ConvLSTM",
    "input_csv": str(CSV_PATH),
    "target_column": target_col,
    "year_column": year_col,
    "month_column": month_col,
    "latitude_column": lat_col,
    "longitude_column": lon_col,
    "feature_columns": feature_columns,
    "input_channels": int(INPUT_CHANNELS),
    "sequence_length": int(SEQUENCE_LENGTH),
    "target_months": TARGET_MONTHS,
    "grid_shape": [int(H), int(W)],
    "latitudes": latitudes.tolist(),
    "longitudes": longitudes.tolist(),
    "train_years": [int(train_years[0]), int(train_years[-1])],
    "validation_years": [int(val_years[0]), int(val_years[-1])],
    "test_years": [int(test_years[0]), int(test_years[-1])],
    "best_config": best_config,
    "target_units_note": (
        "Units are inherited from processed CSV. If tp was standardized upstream, "
        "these values are not physical precipitation units."
    ),
    "test_metrics": test_metrics.to_dict(orient="records")
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
config_path.write_text(json.dumps({
    "seed": SEED,
    "device_used": str(DEVICE),
    "batch_size": BATCH_SIZE,
    "max_epochs_per_candidate": MAX_EPOCHS,
    "patience": PATIENCE,
    "learning_rates_searched": LEARNING_RATES,
    "hidden_channels_searched": HIDDEN_CHANNELS,
    "kernel_sizes_searched": KERNEL_SIZES,
    "selected": best_config
}, indent=2), encoding="utf-8")

print("Saved artifacts:")
for path in [model_path, scalers_path, metadata_path, config_path,
             RESULTS_DIR / "test_metrics.csv",
             RESULTS_DIR / "test_grid_predictions.csv"]:
    if path.exists():
        print(f" - {path.resolve()} ({path.stat().st_size / (1024**2):.2f} MiB)")


## 16. Reload test — verify the saved weights

This final smoke test reconstructs the architecture from saved metadata, loads the weights, and checks that predictions from the reloaded model match the in-memory final model for one held-out batch.

In [ ]:
saved = torch.load(model_path, map_location=DEVICE, weights_only=True)
reloaded_model = ConvLSTMForecaster(
    input_channels=saved["input_channels"],
    hidden_channels=saved["hidden_channels"],
    kernel_size=saved["kernel_size"]
).to(DEVICE)
reloaded_model.load_state_dict(saved["state_dict"])
reloaded_model.eval()
final_model.eval()

batch_x, _batch_y, _batch_mask, _batch_meta = next(iter(test_loader))
batch_x = batch_x.to(DEVICE)

with torch.no_grad():
    pred_original = final_model(batch_x)
    pred_reloaded = reloaded_model(batch_x)

max_abs_difference = float((pred_original - pred_reloaded).abs().max().item())
assert torch.allclose(pred_original, pred_reloaded, atol=1e-6, rtol=1e-5), (
    f"Reloaded model mismatch: max abs difference={max_abs_difference}"
)
print("Reload verification passed.")
print("Maximum absolute prediction difference:", max_abs_difference)


## 17. Artifact inventory and interpretation checklist

Expected output layout:

```text
artifacts/convlstm/
├── models/
│   ├── mahamonsoon_convlstm.pt
│   ├── preprocessing_scalers.joblib
│   ├── model_metadata.json
│   └── training_config.json
├── results/
│   ├── hyperparameter_tuning.csv
│   ├── tuning_histories.json
│   ├── final_training_history.csv
│   ├── test_metrics.csv
│   └── test_grid_predictions.csv
└── figures/
    ├── hyperparameter_validation_curves.png
    ├── test_spatial_field_comparison.png
    ├── test_actual_vs_predicted.png
    └── test_residual_distribution.png
```

Before making scientific claims, verify:
- the precipitation target's physical units and conversion from ERA5 monthly accumulations;
- whether the upstream CSV standardized the target using the full time series (if so, rebuild preprocessing with train-only statistics for strict evaluation);
- whether the grid covers Maharashtra and has been masked/aggregated as intended;
- whether input predictors available at forecast issue time are genuinely known (no future-month predictors);
- whether ConvLSTM improves on a properly constructed climatology/persistence baseline across multiple held-out years.

This notebook provides a complete ConvLSTM training and evaluation pipeline for the available processed grid data, but model quality must be judged from the actual run's test results and domain validation—not from architecture complexity alone.